In [1]:
# For debugging
%load_ext autoreload
%autoreload 2

In [ ]:
from copy import deepcopy

import torch
import torch.nn
import torch.optim
from torch.utils.data import DataLoader

import xarray as xr
import numpy

from tqdm.notebook import tqdm 

from neural_net import get_net
from constants import *

import matplotlib.pyplot as plt

In [3]:
torch.manual_seed(42)

# Settings

In [4]:
device = torch.device("cuda")
dtype = torch.bfloat16

In [5]:
#batch_size = 512
#n_epochs = 20
#n_layers = 4

batch_size = 8                # small to avoid OOM; raise if memory allows
n_epochs = 1
n_layers = 2
n_features = 64

# Load data

In [6]:
ds_train = xr.open_zarr("../data/sqg_train.zarr")["q"].compute(num_workers=4) # training data 

# Normalize data
train_data = torch.cat((
    (torch.as_tensor(ds_train.values[:-1], dtype=dtype)-in_mean) / in_std,
    (torch.as_tensor(ds_train.values[1:]-ds_train.values[:-1], dtype=dtype)-res_mean) / res_std
), dim=1)

del ds_train

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

In [7]:
ds_val = xr.open_zarr("../data/sqg_val.zarr")["q"].compute(num_workers=16) # validation data 
val_data = torch.cat((
    (torch.as_tensor(ds_val.values[:-1], dtype=dtype)-in_mean) / in_std,
    (torch.as_tensor(ds_val.values[1:]-ds_val.values[:-1], dtype=dtype)-res_mean) / res_std
), dim=1)
del ds_val

val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

In [8]:
weights_lat = torch.as_tensor(weights_lat, device=device, dtype=dtype).squeeze()

# Define neural network

Use of a convolutional neural network

In [9]:
pbar_epoch = tqdm(range(n_epochs))

mse_val = torch.inf
best_mse = torch.inf
best_model = None

for _ in pbar_epoch:
    pbar_train = tqdm(iter(train_loader), total=len(train_loader), leave=True)
    # Training loop
    convnet = convnet.train()
    for batch in pbar_train:        
        batch = batch.to(device=device, dtype=dtype)
        data_in, data_target = batch.split((7, 7), dim=1)

        # Change for flow matching model
        ## Sample epsilon (z_0) as random draw from a normal distribution
        noise = torch.randn_like(data_target)

        ## Stratified sampling of pseudo time to reduce variance during optimisation (Kingma et al., 2022)
        ### Take pseudo time between [0, 1)
        pseudo_time = torch.linspace(0, 1, batch.shape[0]+1, device=device, dtype=dtype)[:-1, None]
        ### Introduce random shift between 0 and 1
        time_shift = torch.rand(1, device=device, dtype=dtype)
        pseudo_time = pseudo_time + time_shift
        ### Ensure that pseudo time is between 0 and 1
        pseudo_time = pseudo_time%1

        ## Construct intermediate state (z_t) with linear interpolant
        intermediate_state = pseudo_time[..., None, None] * data_target \
            + (1-pseudo_time[..., None, None]) * noise
        target_velocity = data_target-noise

        ## Neural network input = [intermediate state, initial conditions]
        input_tensor = torch.cat((
            intermediate_state, data_in
        ), dim=1)                                 

        optim.zero_grad()
        ## Neural network predicts velocity now
        prediction = convnet(input_tensor, pseudo_time)
        error = (prediction - target_velocity).pow(2)

        ## Reweight for latitudes
        mse_train = (weights_lat * error).mean()
        mse_train.backward()
        optim.step()

        pbar_train.set_postfix(mse_train=mse_train.item(), mse_val=mse_val)

    mse_val = 0
    samples_val = 0
    pbar_val = tqdm(enumerate(val_loader), total=len(val_loader), leave=False)
    
    # Validation loop
    convnet = convnet.eval()
    for k, batch in pbar_val:        
        batch = batch.to(device=device, dtype=dtype)
        data_in, data_target = batch.split((7, 7), dim=1)

        # Change for flow matching model
        ## Sample epsilon (z_0) as random draw from a normal distribution
        noise = torch.randn_like(data_target)

        ## Stratified sampling of pseudo time to reduce variance during optimisation (Kingma et al., 2022)
        ### Take pseudo time between [0, 1)
        pseudo_time = torch.linspace(0, 1, batch.shape[0]+1, device=device, dtype=dtype)[:-1, None]
        ### Introduce random shift between 0 and 1
        time_shift = torch.rand(1, device=device, dtype=dtype)
        pseudo_time = pseudo_time + time_shift
        ### Ensure that pseudo time is between 0 and 1
        pseudo_time = pseudo_time%1

        ## Construct intermediate state (z_t) with linear interpolant
        intermediate_state = pseudo_time[..., None, None] * data_target \
            + (1-pseudo_time[..., None, None]) * noise
        target_velocity = data_target-noise

        ## Neural network input = [intermediate state, initial conditions]
        input_tensor = torch.cat((
            intermediate_state, data_in
        ), dim=1)         
        
        with torch.no_grad():
            prediction = convnet(input_tensor, pseudo_time)
        error = (prediction - target_velocity).pow(2)
        curr_se = (weights_lat * error).mean(dim=(1, 2, 3)).sum().item()
        mse_val = mse_val * samples_val + curr_se
        samples_val = samples_val + len(batch)
        mse_val = mse_val / samples_val

    pbar_train.set_postfix(mse_train=mse_train.item(), mse_val=mse_val)
        
    # Check if new model is better
    if mse_val < 0.999 * best_mse: # 0.999 to get rid of randomness 
        best_mse = mse_val
        best_model = deepcopy(convnet).cpu()

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/25 [00:00<?, ?it/s]

NameError: name 'convnet' is not defined

In [10]:
convnet = get_net(
    # Increase of number of input channels intermediate state + initial conditions
    n_input=14,
    n_output=7, n_layers=n_layers, n_features=256, mult=2,
    # Activation of pseudo time
    use_time=True, wave_length=0.1,
    device=device, dtype=dtype
)
optim = torch.optim.Adam(convnet.parameters(), lr=1E-3)

# Store best model

In [19]:
state_dict = best_model.cpu().state_dict()
torch.save(state_dict, "/libre/finnt/data/teaching/flow_model.ckpt")